# 面试问题：LLM 流式输出怎样正确处理跨 token 的 stop sequence，并避免把停止串泄露给用户？

**一句话回答。** 不要对每个 chunk 独立 substring 搜索：应保留所有可能继续组成 stop string 的后缀，只有确定不可能匹配的字符才能提交给客户端；命中时不提交 stop string，结束时再 flush pending 内容。

本 Notebook 仅用 Python 标准库手写数据合同、核心算法和失败分支；受控样例用于验证不变量，不代表生产吞吐、模型质量或硬件精度。

**资料入口。** [Transformers StopStringCriteria](https://huggingface.co/docs/transformers/en/internal/generation_utils) 说明 stop string 需要匹配 tokenizer 造成的跨 token overhang；本例在已正确解码的字符流上实现提交状态机。


In [ ]:
question = "流式 stop sequence"  # 执行本行的状态、计算或校验逻辑。
assert "stop" in question  # 执行本行的状态、计算或校验逻辑。
assert len("<END>") == 5  # 执行本行的状态、计算或校验逻辑。
assert True  # 执行本行的状态、计算或校验逻辑。

## 1. stop string 是服务协议，不是普通字符串替换

生成器、tokenizer、UTF-8 增量解码器和 SSE/HTTP 输出的边界都不同。服务端必须声明是否返回 stop string、EOS/长度上限优先级和 stop 配置版本；本例先假设输入是合法 Unicode 字符 chunk。


In [ ]:
stops = ("<END>", "<STOP>")  # 执行本行的状态、计算或校验逻辑。
state = {"pending": "", "emitted": "", "stopped": False, "matched": None}  # 执行本行的状态、计算或校验逻辑。
assert all(stop for stop in stops)  # 执行本行的状态、计算或校验逻辑。
assert state["pending"] == ""  # 执行本行的状态、计算或校验逻辑。
assert not state["stopped"]  # 执行本行的状态、计算或校验逻辑。

## 2. 只保留仍可能成为 stop 前缀的最长后缀

每接收一个字符后，若没有完整命中，就找 candidate 的最长后缀，使其为任意 stop 的真前缀。其余字符已经不可能参与未来匹配，可以安全发送，因而不会在下一 chunk 到来时需要回滚用户已见内容。


In [ ]:
def reserve_suffix(text, stop_values):  # 执行本行的状态、计算或校验逻辑。
    candidates = [text[-size:] for size in range(1, len(text) + 1) if any(text[-size:] == stop[:size] and size < len(stop) for stop in stop_values)]  # 执行本行的状态、计算或校验逻辑。
    return max(candidates, key=len) if candidates else ""  # 执行本行的状态、计算或校验逻辑。
assert reserve_suffix("回答<EN", stops) == "<EN"  # 执行本行的状态、计算或校验逻辑。
assert reserve_suffix("普通文字", stops) == ""  # 执行本行的状态、计算或校验逻辑。
assert reserve_suffix("<", stops) == "<"  # 执行本行的状态、计算或校验逻辑。

## 3. 逐字符状态机，命中前不提交潜在停止串

字符级循环是 tokenizer 边界无关的教学表达。生产可在 token-byte level 编译 matcher，但必须保持同一语义：先检测完整匹配，再提交安全前缀；多个 stop 同时命中时记录最长规则以便审计。


In [ ]:
def feed(current, chunk, stop_values):  # 执行本行的状态、计算或校验逻辑。
    for char in chunk:  # 执行本行的状态、计算或校验逻辑。
        candidate = current["pending"] + char  # 执行本行的状态、计算或校验逻辑。
        matches = [stop for stop in stop_values if candidate.endswith(stop)]  # 执行本行的状态、计算或校验逻辑。
        if matches:  # 执行本行的状态、计算或校验逻辑。
            current["stopped"] = True  # 执行本行的状态、计算或校验逻辑。
            current["matched"] = max(matches, key=len)  # 执行本行的状态、计算或校验逻辑。
            current["pending"] = ""  # 执行本行的状态、计算或校验逻辑。
            return current  # 执行本行的状态、计算或校验逻辑。
        reserved = reserve_suffix(candidate, stop_values)  # 执行本行的状态、计算或校验逻辑。
        current["emitted"] += candidate[:-len(reserved)] if reserved else candidate  # 执行本行的状态、计算或校验逻辑。
        current["pending"] = reserved  # 执行本行的状态、计算或校验逻辑。
    return current  # 执行本行的状态、计算或校验逻辑。
assert callable(feed)  # 执行本行的状态、计算或校验逻辑。
assert state["matched"] is None  # 执行本行的状态、计算或校验逻辑。
assert len(stops) == 2  # 执行本行的状态、计算或校验逻辑。

## 4. 跨 chunk 命中也不能泄露

这里停止串被切成三个 chunk。`<` 与 `<END` 一直留在 pending；最后一个 `>` 才触发停止。停止后的同一网络 chunk 里的后续字符也不得继续暴露，需要由上层取消 decode/忽略尾部。


In [ ]:
for chunk in ("答案<", "END", ">后续"):  # 执行本行的状态、计算或校验逻辑。
    state = feed(state, chunk, stops)  # 执行本行的状态、计算或校验逻辑。
    if state["stopped"]:  # 执行本行的状态、计算或校验逻辑。
        break  # 执行本行的状态、计算或校验逻辑。
assert state["stopped"]  # 执行本行的状态、计算或校验逻辑。
assert state["matched"] == "<END>"  # 执行本行的状态、计算或校验逻辑。
assert state["emitted"] == "答案"  # 执行本行的状态、计算或校验逻辑。

## 5. 没有命中时，在 finish 时 flush pending

pending 只是为了等待未来 chunk；正常 EOS、max_tokens 或网络结束时必须 flush，否则像结尾的 `<E` 会被无故吞掉。flush 是 terminal 事件的一部分，且只能调用一次。


In [ ]:
def finish(current):  # 执行本行的状态、计算或校验逻辑。
    if not current["stopped"]:  # 执行本行的状态、计算或校验逻辑。
        current["emitted"] += current["pending"]  # 执行本行的状态、计算或校验逻辑。
    current["pending"] = ""  # 执行本行的状态、计算或校验逻辑。
    return current  # 执行本行的状态、计算或校验逻辑。
unfinished = {"pending": "<E", "emitted": "文本", "stopped": False, "matched": None}  # 执行本行的状态、计算或校验逻辑。
finished = finish(unfinished)  # 执行本行的状态、计算或校验逻辑。
assert finished["emitted"] == "文本<E"  # 执行本行的状态、计算或校验逻辑。
assert finished["pending"] == ""  # 执行本行的状态、计算或校验逻辑。
assert not finished["stopped"]  # 执行本行的状态、计算或校验逻辑。

## 6. 重叠和多个 stop 的顺序应可解释

短 stop 是长 stop 的前缀时，最早完成的位置决定停止；同一位置多个候选命中时记录最长值是审计规则，不改变已提交文本。禁止空 stop，否则每个字符前都可“命中”。


In [ ]:
overlap_stops = ("##", "###")  # 执行本行的状态、计算或校验逻辑。
overlap = {"pending": "", "emitted": "", "stopped": False, "matched": None}  # 执行本行的状态、计算或校验逻辑。
overlap = feed(overlap, "标题###尾部", overlap_stops)  # 执行本行的状态、计算或校验逻辑。
assert overlap["stopped"]  # 执行本行的状态、计算或校验逻辑。
assert overlap["matched"] == "##"  # 执行本行的状态、计算或校验逻辑。
assert overlap["emitted"] == "标题"  # 执行本行的状态、计算或校验逻辑。

## 7. 协议事件要区分 delta、finish 和 stop 原因

客户端不应靠“最后一个 delta 是否为空”猜测结束。finish event 应携带 sequence、reason、matched stop 的非敏感标识和配置版本；是否 include stop string 是请求参数，默认应为 false。


In [ ]:
event = {"sequence": 3, "type": "finish", "reason": "stop", "matched": state["matched"], "include_stop": False, "stop_config": "v1"}  # 执行本行的状态、计算或校验逻辑。
assert event["type"] == "finish"  # 执行本行的状态、计算或校验逻辑。
assert event["reason"] == "stop"  # 执行本行的状态、计算或校验逻辑。
assert event["matched"] not in state["emitted"]  # 执行本行的状态、计算或校验逻辑。
assert not event["include_stop"]  # 执行本行的状态、计算或校验逻辑。

## 8. 输入校验和生产边界

空字符串、过长 stop 集、未完成 UTF-8 字节、tool call JSON 与 max/min token 的优先级都需要独立测试。本例仅验证字符流；真实服务必须使用 tokenizer-aware matcher，并在终止时停止 token 生成与 KV 增长。


In [ ]:
def valid_stops(stop_values, limit):  # 执行本行的状态、计算或校验逻辑。
    return bool(stop_values) and all(isinstance(stop, str) and stop for stop in stop_values) and sum(len(stop) for stop in stop_values) <= limit  # 执行本行的状态、计算或校验逻辑。
assert valid_stops(stops, 32)  # 执行本行的状态、计算或校验逻辑。
assert not valid_stops(("",), 32)  # 执行本行的状态、计算或校验逻辑。
assert not valid_stops(("x" * 33,), 32)  # 执行本行的状态、计算或校验逻辑。

## 面试收束

面试中要强调“检测到 stop”与“安全地流式提交”是两件事：后者需要 pending buffer/前缀状态机。再补充 tokenizer overhang、UTF-8 增量解码、finish reason、include-stop 语义、取消生成和配置版本，答案才覆盖真正的服务边界。
